# CS-552: Modern NLP — Exercise: Prompting Techniques
## Chain-of-Thought, In-Context Learning, and Self-Consistency

**Objectives:**
- Understand and implement **Chain-of-Thought (CoT)** prompting — eliciting step-by-step reasoning from LLMs.
- Understand the role of **In-Context Learning (ICL)** — how providing worked examples in the prompt improves CoT.
- Understand and implement **Self-Consistency** — sampling multiple reasoning paths and aggregating via majority vote.
- Observe how each technique *progressively* improves a small LLM's accuracy on the same task.

**Setup:** This notebook runs on **Google Colab Free** with a **T4 GPU**. We use `Qwen/Qwen2.5-0.5B-Instruct` (0.5B parameters). This model is intentionally small so that improvements from prompting techniques are clearly visible.

**Task:** All exercises use **multi-step arithmetic word problems** — a task where small models
frequently make errors, and where each prompting technique provides a measurable boost.

**Expected progression:**
```
Zero-shot  <  Zero-shot CoT  <  Few-shot CoT  <  Self-Consistency
```

> ⏱️ **Estimated runtime:** ~25–35 minutes end-to-end.

## 0. Environment Setup

In [ ]:
# Check GPU availability
import torch
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"✅ GPU available: {gpu.name}")
    print(f"   Memory: {gpu.total_memory / 1024**3:.1f} GB")
else:
    print("❌ No GPU found! Go to Runtime > Change runtime type > T4 GPU")

In [ ]:
!pip install -q transformers accelerate

In [ ]:
import torch
import re
import random
from collections import Counter
from transformers import AutoModelForCausalLM, AutoTokenizer

random.seed(42)
torch.manual_seed(42)

## 1. Loading the Model

We load `Qwen2.5-0.5B-Instruct` in float16. This instruction-tuned model can follow instructions
but is weak enough at multi-step arithmetic that prompting techniques make a real difference.

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
)
print(f"Model loaded on: {model.device}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.0f}M")

### Helper: Generation Function

This utility wraps the chat template and generation call. You'll use it throughout.

In [ ]:
def generate(messages, max_new_tokens=256, temperature=0.0, do_sample=False):
    """Generate a response given a list of chat messages.

    Args:
        messages: list of dicts with 'role' and 'content' keys.
        max_new_tokens: max tokens to generate.
        temperature: sampling temperature (0 = greedy).
        do_sample: whether to sample (must be True if temperature > 0).

    Returns:
        str: the model's response text.
    """
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature if do_sample else None,
            do_sample=do_sample,
            pad_token_id=tokenizer.eos_token_id,
        )

    # Decode only the newly generated tokens
    response = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return response.strip()

## 2. Dataset & Evaluation

We define a shared set of multi-step arithmetic word problems (3–4 reasoning steps each),
along with demonstration examples for few-shot CoT and helper functions for evaluation.

The problems involve fractions, percentages, and chained operations — designed to be hard enough
that the 0.5B model can't reliably solve them without step-by-step reasoning.

In [ ]:
# ============================================================
# Demonstration examples — used for few-shot CoT (Part 2).
# Each has a question, step-by-step reasoning, and final answer.
# ============================================================

math_demos = [
    {
        "question": "A baker made 60 cupcakes. She sold half in the morning, gave away 8 to neighbors, and then made 15 more in the afternoon. How many cupcakes does she have?",
        "reasoning": "She started with 60 cupcakes. She sold half: 60 / 2 = 30 remaining. She gave away 8: 30 - 8 = 22 remaining. She made 15 more: 22 + 15 = 37 cupcakes.",
        "answer": 37
    },
    {
        "question": "Tom earns $12 per hour. He worked 5 hours on Monday and 7 hours on Tuesday. He spent $40 on groceries. How much money does he have left from these two days?",
        "reasoning": "Monday earnings: 5 × $12 = $60. Tuesday earnings: 7 × $12 = $84. Total earnings: $60 + $84 = $144. After groceries: $144 - $40 = $104.",
        "answer": 104
    },
    {
        "question": "A school has 4 classes. Each class has 28 students. If 15 students are absent across all classes, how many students are present?",
        "reasoning": "Total students: 4 × 28 = 112. Students present: 112 - 15 = 97.",
        "answer": 97
    },
]

# ============================================================
# Test problems — 3-4 step reasoning, with fractions/percentages
# ============================================================

math_test = [
    {"question": "A store had 85 shirts. They sold 1/5 of them on Monday, then received a shipment of 30 shirts, then sold 18 more on Tuesday. How many shirts are left?", "answer": 80},
    {"question": "Lisa has $120. She buys 4 books at $12 each and 3 pens at $3 each. She then earns $25 from babysitting. How much money does she have now?", "answer": 88},
    {"question": "A parking garage has 5 floors. Each floor has 32 spaces. If 78 cars are parked and 14 more arrive, how many empty spaces remain?", "answer": 68},
    {"question": "A farmer collects 24 eggs per day from his chickens. He sells 2/3 of them and keeps the rest. How many eggs does he keep in a week (7 days)?", "answer": 56},
    {"question": "A train has 8 carriages. Each carriage has 45 seats. If 217 passengers board and 38 get off at the first stop, how many empty seats are there?", "answer": 181},
    {"question": "Maria baked 3 batches of cookies. Each batch makes 24 cookies. She gave 1/4 of all cookies to her friend and ate 5 herself. How many cookies are left?", "answer": 49},
    {"question": "A company has 250 employees. 60% work in the office and the rest work remotely. If 15 remote workers switch to office work, how many remote workers remain?", "answer": 85},
    {"question": "Jake reads 35 pages per day. After reading for 6 days, he realizes the book has 300 pages total. How many pages does he have left?", "answer": 90},
    {"question": "A warehouse stores boxes in stacks of 12. They have 15 full stacks and one partial stack of 7 boxes. If they ship out 50 boxes, how many remain?", "answer": 137},
    {"question": "A school cafeteria prepares 200 lunches. 3/8 are vegetarian and the rest are non-vegetarian. If 20 vegetarian lunches are left over, how many vegetarian lunches were eaten?", "answer": 55},
]

print(f"Demonstration examples: {len(math_demos)}")
print(f"Test problems: {len(math_test)}")

In [ ]:
def extract_number(response):
    """Extract the final numerical answer from a model response."""
    response = response.replace(",", "")

    # Pattern: 'the answer is X'
    match = re.search(r'(?:the answer is|answer is|answer:)\s*\$?([\d.]+)', response.lower())
    if match:
        try:
            return int(float(match.group(1)))
        except ValueError:
            pass

    # Pattern: '= X' (last occurrence)
    matches = re.findall(r'=\s*\$?([\d.]+)', response)
    if matches:
        try:
            return int(float(matches[-1]))
        except ValueError:
            pass

    # Fallback: last number in the text
    numbers = re.findall(r'\b(\d+)\b', response)
    if numbers:
        try:
            return int(numbers[-1])
        except ValueError:
            pass
    return None


def evaluate_math(prompting_fn, test_data, show_reasoning=False):
    """Evaluate a math prompting function on test problems."""
    results, correct = [], 0
    for item in test_data:
        raw = prompting_fn(item["question"])
        pred = extract_number(raw)
        is_correct = pred == item["answer"]
        correct += is_correct
        results.append((item["question"], pred, item["answer"], raw, is_correct))
    acc = correct / len(test_data)

    print(f"Accuracy: {acc:.1%}")
    print("=" * 70)
    for q, pred, gold, raw, ok in results:
        mark = "✅" if ok else "❌"
        print(f"{mark} Gold={gold:4d} | Pred={str(pred):>6s} | {q[:55]}")
        if show_reasoning:
            print(f"   → {raw[:130]}...")
        elif not ok:
            print(f"   Raw: {raw[:80]}")

    return acc, results

---
## Part 1: Chain-of-Thought Prompting

### Background

**Chain-of-Thought (CoT) prompting** (Wei et al., 2022) improves LLM reasoning by asking the model
to *show its work* — generating intermediate reasoning steps before the final answer.

**Why it works:** When a model is asked to directly produce an answer to a multi-step problem, it
must perform all the reasoning in a single forward pass (i.e., within its hidden states). By
generating intermediate steps as tokens, the model gets additional "computation" — each new token
can attend to the previously generated reasoning, effectively allowing the model to decompose complex
problems into simpler sub-problems.

The simplest form is **zero-shot CoT** (Kojima et al., 2022): just append *"Let's think step by
step"* to the prompt. No examples needed — just the instruction to reason explicitly.

In this part, we compare:
- **Zero-shot (direct):** Ask for the answer only — no reasoning.
- **Zero-shot CoT:** Ask the model to reason step by step before answering.

### Exercise 1a: Zero-Shot Prompting (Baseline)

Implement a zero-shot prompting function that asks the model to solve a math word problem
directly, returning only the numerical answer. No examples, no reasoning.

In [ ]:
def zero_shot_math(question):
    """
    Solve a math word problem using zero-shot prompting (no examples, no reasoning).

    Instructions:
    - Build a `messages` list with a single user message.
    - Ask the model to solve the problem and give ONLY the numerical answer.
    - Call generate(messages, max_new_tokens=32) and return the result.

    Args:
        question (str): A math word problem.
    Returns:
        str: The model's raw response.
    """
    messages = [
        {
            "role": "user",
            "content": (
                "Solve this math problem. Reply with ONLY the final numerical answer, "
                "nothing else.\n\n"
                f"Problem: {question}\n\n"
                "Answer:"
            )
        }
    ]
    return generate(messages, max_new_tokens=32)

In [ ]:
print("=== Zero-Shot (no reasoning) ===")
zero_shot_acc, zero_shot_results = evaluate_math(zero_shot_math, math_test)

### Exercise 1b: Zero-Shot Chain-of-Thought

Adding "Let's think step by step" to the prompt is a remarkably simple but effective trick.
No examples are needed — just the instruction to reason explicitly.

**Important:** Since the model now produces a longer reasoning chain:
1. Allow more tokens (`max_new_tokens=512`).
2. Ask the model to clearly state "The answer is <number>" at the end so we can parse it.

In [ ]:
def zero_shot_cot_math(question):
    """
    Solve a math problem using zero-shot chain-of-thought.

    Instructions:
    - Include "Let's think step by step" in the prompt.
    - Ask the model to end with "The answer is <number>".
    - Use max_new_tokens=512 to allow space for reasoning.

    Args:
        question (str): A math word problem.
    Returns:
        str: The model's response (including reasoning).
    """
    messages = [
        {
            "role": "user",
            "content": (
                f"Solve this math problem step by step. Let's think step by step. "
                f"At the end, state your final answer as: \"The answer is <number>\".\n\n"
                f"Problem: {question}"
            )
        }
    ]
    return generate(messages, max_new_tokens=512)

In [ ]:
print("=== Zero-Shot CoT ===")
zs_cot_acc, zs_cot_results = evaluate_math(zero_shot_cot_math, math_test, show_reasoning=True)

In [ ]:
print("\n" + "=" * 50)
print("  PART 1 RESULTS")
print("=" * 50)
print(f"  Zero-shot (direct): {zero_shot_acc:.1%}")
print(f"  Zero-shot CoT:      {zs_cot_acc:.1%}")
print(f"  Improvement:        {zs_cot_acc - zero_shot_acc:+.1%}")
print("=" * 50)

### 🤔 Discussion Questions (Part 1)

1. **How big was the improvement from CoT?** The model has the same parameters — why does
   generating intermediate tokens help so much?
2. **Examine the reasoning chains.** Are the intermediate steps correct? Does the model ever reach
   the right answer through flawed reasoning, or vice versa?
3. **Failure cases:** Look at problems where zero-shot CoT still fails. What goes wrong —
   arithmetic errors, misunderstanding the problem, or something else?

---
## Part 2: Few-Shot Chain-of-Thought (ICL + CoT)

### Background

**Few-shot CoT** (Wei et al., 2022) combines **In-Context Learning** with chain-of-thought.
You provide demonstration examples that include both the **reasoning steps** and the **final answer**.

**In-Context Learning (ICL)** is the ability of LLMs to learn from examples in the prompt without
any gradient updates (Brown et al., 2020). In the few-shot CoT setting, the demonstrations serve
double duty:
1. They show the model **what task** to solve (like regular ICL).
2. They show the model **how to reason** — the structured step-by-step format teaches the model
   to decompose problems and chain computations correctly.

**Why this helps beyond zero-shot CoT:** When the model generates its own reasoning structure from
scratch (zero-shot CoT), it often makes formatting errors, skips steps, or loses track of
intermediate results. The demonstrations provide a *template* for clean reasoning, reducing these
errors.

### Exercise 2: Few-Shot CoT

Implement few-shot chain-of-thought prompting using the demonstration examples from `math_demos`.
Each demo includes a question, reasoning chain, and answer.

In [ ]:
def few_shot_cot_math(question):
    """
    Solve a math problem using few-shot chain-of-thought.

    Instructions:
    - Use examples from `math_demos` (each has 'question', 'reasoning', 'answer').
    - Format each demo like:
        Q: <question>
        A: Let's think step by step. <reasoning> The answer is <answer>.
    - After the demos, add the test question in the same Q/A format,
      ending with "A: Let's think step by step." so the model continues.
    - Use max_new_tokens=512.

    Args:
        question (str): A math word problem.
    Returns:
        str: The model's response.
    """
    prompt = ""
    for ex in math_demos:
        prompt += f"Q: {ex['question']}\n"
        prompt += f"A: Let's think step by step. {ex['reasoning']} The answer is {ex['answer']}.\n\n"
    prompt += f"Q: {question}\nA: Let's think step by step."

    messages = [{"role": "user", "content": prompt}]
    return generate(messages, max_new_tokens=512)

In [ ]:
print("=== Few-Shot CoT ===")
fs_cot_acc, fs_cot_results = evaluate_math(few_shot_cot_math, math_test, show_reasoning=True)

In [ ]:
print("\n" + "=" * 50)
print("  PART 2 RESULTS")
print("=" * 50)
print(f"  Zero-shot (direct): {zero_shot_acc:.1%}")
print(f"  Zero-shot CoT:      {zs_cot_acc:.1%}")
print(f"  Few-shot CoT:       {fs_cot_acc:.1%}")
print("=" * 50)

### 🤔 Discussion Questions (Part 2)

1. **Compare the reasoning chains** from zero-shot CoT vs few-shot CoT. Is the few-shot version
   more structured? Does it make fewer arithmetic mistakes?
2. **Which problems did few-shot CoT fix** that zero-shot CoT got wrong? What about the reverse?
3. **Role of the demonstrations:** The demos show a specific reasoning *format* (short, chained
   equations). How much of the improvement comes from the format vs. the content?
4. **What if the demos had wrong reasoning?** Would the model copy the errors? (This is a real
   concern in practice — demo quality matters!)

---
## Part 3: Self-Consistency

### Background

**Self-Consistency** (Wang et al., 2023) improves chain-of-thought by sampling *multiple* reasoning
paths and taking a **majority vote** on the final answer.

**Key insight:** A single CoT chain can go wrong at any step — an arithmetic slip, a misread
condition, etc. But if you sample many chains (with temperature > 0), the correct answer tends to
appear more frequently than any particular wrong answer. This is because there are many ways to
arrive at a wrong answer (different errors at different steps), but typically fewer ways to get the
right one.

**Algorithm:**
1. Use the same CoT prompt (here: few-shot CoT from Part 2).
2. Sample N responses with temperature > 0 (`do_sample=True`).
3. Extract the numerical answer from each response.
4. Return the most common answer (majority vote).

**Why it works:** This is an ensemble method over reasoning paths. Each sample explores a different
trajectory through the reasoning space. Errors tend to be random (spread across wrong answers),
while correct reasoning converges (clusters on the right answer).

**Trade-off:** N× more compute per question, but accuracy gains are often substantial.

### Exercise 3a: Implementing Self-Consistency

Implement the self-consistency method:
1. Reuse the few-shot CoT prompt from Exercise 2.
2. Sample N responses with `temperature > 0` and `do_sample=True`.
3. Extract the numerical answer from each.
4. Return the majority vote (most common answer).

In [ ]:
def self_consistency_math(question, n_samples=15, temperature=0.7):
    """
    Solve a math problem using self-consistency (majority vote over CoT samples).

    Instructions:
    - Build the same few-shot CoT prompt as in Exercise 2.
    - Generate `n_samples` responses with do_sample=True and the given temperature.
    - Extract the numerical answer from each using extract_number().
    - Filter out None (failed extractions).
    - Return (majority_answer, list_of_individual_answers).

    Args:
        question (str): A math word problem.
        n_samples (int): Number of reasoning paths to sample.
        temperature (float): Sampling temperature.
    Returns:
        tuple: (majority_answer: int|None, individual_answers: list[int|None])
    """
    # Build few-shot CoT prompt (same as Exercise 2)
    prompt = ""
    for ex in math_demos:
        prompt += f"Q: {ex['question']}\n"
        prompt += f"A: Let's think step by step. {ex['reasoning']} The answer is {ex['answer']}.\n\n"
    prompt += f"Q: {question}\nA: Let's think step by step."

    messages = [{"role": "user", "content": prompt}]

    # Sample multiple responses
    individual_answers = []
    for _ in range(n_samples):
        response = generate(messages, max_new_tokens=512, temperature=temperature, do_sample=True)
        answer = extract_number(response)
        individual_answers.append(answer)

    # Majority vote (excluding None)
    valid = [a for a in individual_answers if a is not None]
    if not valid:
        return None, individual_answers

    counter = Counter(valid)
    majority = counter.most_common(1)[0][0]
    return majority, individual_answers

In [ ]:
# Evaluate self-consistency (takes a few minutes due to multiple samples per question)
print("Running self-consistency (15 samples per question)...")
print("This will take a few minutes — each question requires 15 forward passes.\n")

sc_correct = 0
sc_results = []

for item in math_test:
    majority, individual = self_consistency_math(item["question"], n_samples=15, temperature=0.7)
    gold = item["answer"]
    is_correct = majority == gold
    sc_correct += is_correct
    sc_results.append((item["question"], majority, gold, individual, is_correct))

    mark = "✅" if is_correct else "❌"
    valid = [a for a in individual if a is not None]
    counter = Counter(valid)
    print(f"{mark} Gold={gold:4d} | Majority={str(majority):>6s} | Votes: {dict(counter)}")
    print(f"   {item['question'][:70]}")

sc_acc = sc_correct / len(math_test)
print(f"\nSelf-Consistency Accuracy: {sc_acc:.1%}")

### Exercise 3b: Analyzing Vote Distributions

Let's visualize how the votes are distributed. Look for patterns:
- **High agreement** (e.g., 12/15 for one answer) → the model is confident and likely correct.
- **Split votes** → the model is uncertain; majority vote helps by picking the plurality winner.

In [ ]:
print("=" * 60)
print("  VOTE DISTRIBUTION ANALYSIS")
print("=" * 60)

for q, majority, gold, individual, ok in sc_results:
    valid = [a for a in individual if a is not None]
    counter = Counter(valid)
    total = len(valid)

    mark = "✅" if ok else "❌"
    print(f"\n{mark} Q: {q[:65]}...")
    print(f"   Gold: {gold} | Majority: {majority} | Agreement: {counter.get(majority, 0)}/{total}")

    for ans, count in counter.most_common():
        bar = "█" * count
        label = " ← correct" if ans == gold else ""
        print(f"   {ans:>6d}: {bar} ({count}){label}")

---
## Final Comparison

In [ ]:
print("=" * 55)
print("  FINAL RESULTS: PROMPTING TECHNIQUES PROGRESSION")
print("=" * 55)
print(f"  1. Zero-shot (direct):    {zero_shot_acc:6.1%}")
print(f"  2. Zero-shot CoT:         {zs_cot_acc:6.1%}")
print(f"  3. Few-shot CoT:          {fs_cot_acc:6.1%}")
print(f"  4. Self-Consistency (15): {sc_acc:6.1%}")
print(f"  " + "-" * 40)
print(f"  Total improvement:        {sc_acc - zero_shot_acc:+5.1%}")
print("=" * 55)

### 🤔 Final Discussion Questions

1. **Did the results follow the expected ordering?**
   `Zero-shot < Zero-shot CoT < Few-shot CoT < Self-Consistency`
   If not, which technique didn't help as much as expected? Why?

2. **Diminishing returns:** Compare the jump from each step to the next.
   Where is the biggest gain? Where do returns diminish?

3. **Cost-accuracy tradeoff:** Self-consistency used 15× the compute of a single CoT pass.
   Was the accuracy gain worth it? In what scenarios would you pay this cost?

4. **The role of reasoning demonstrations:** Few-shot CoT improved over zero-shot CoT by providing
   a reasoning *template*. What does this tell us about how LLMs "reason"?

5. **When do you need which technique?**
   - CoT: When the task requires multi-step reasoning.
   - Few-shot CoT: When you also want to control the reasoning *format*.
   - Self-consistency: When you need high reliability and can afford more compute.
   Can you think of real-world scenarios for each?

---
## Bonus Experiments

### Bonus 1: ICL Without CoT

Does showing the model **solved examples** (question → answer, *without reasoning steps*) help?
Implement few-shot prompting where each demo only shows the question and the final answer.
Compare this to zero-shot and to few-shot CoT to isolate the effect of demonstrations vs. reasoning.

```python
def few_shot_no_cot(question, k=3):
    """Few-shot ICL: show Q→A pairs without reasoning."""
    # YOUR CODE HERE
    # Hint: Format each demo as:
    #   Q: <question>
    #   A: <answer>
    # (no reasoning steps!)
    pass
```

**Discussion:** For these multi-step math problems, you'll likely find that ICL alone (without CoT)
doesn't help much — the bottleneck is *reasoning*, not *task understanding*. But for tasks like
classification or formatting, ICL alone can be very effective. The key insight: **ICL and CoT address**
**different bottlenecks** — ICL helps with task specification, CoT helps with reasoning.

### Bonus 2: Temperature Sweep for Self-Consistency

Run self-consistency with temperature ∈ {0.3, 0.5, 0.7, 1.0} and N=15.
Plot accuracy vs temperature. Too low → all samples agree (no diversity, no benefit over greedy).
Too high → all samples are noisy (no signal in the majority vote). What's the sweet spot?

### Bonus 3: Varying N for Self-Consistency

Run self-consistency with N ∈ {1, 3, 5, 10, 15, 25} at temperature=0.7.
Plot accuracy vs N. At what point do more samples stop helping?

### Bonus 4: Model Scaling

Try `Qwen/Qwen2.5-1.5B-Instruct` or `Qwen/Qwen2.5-3B-Instruct` (both fit on T4 in float16).
How does the gap between techniques change with model size? At what size do prompting techniques
become unnecessary for these problems?

### References

- Brown et al. (2020). *Language Models are Few-Shot Learners.* NeurIPS.
- Wei et al. (2022). *Chain-of-Thought Prompting Elicits Reasoning in Large Language Models.* NeurIPS.
- Kojima et al. (2022). *Large Language Models are Zero-Shot Reasoners.* NeurIPS.
- Wang et al. (2023). *Self-Consistency Improves Chain of Thought Reasoning in Language Models.* ICLR.
- Dai et al. (2023). *Why Can GPT Learn In-Context?* ACL.